In [1]:
import pandas as pd
import pyodbc
import re
import os
import pyarrow
import numpy as np

# ─────────────────────────────────────────
# CONFIGURAÇÃO
# ─────────────────────────────────────────
quem_esta_usando = os.getlogin()
credenciais = pd.read_csv(
    fr"C:\Users\{quem_esta_usando}\Documents\acesso_banco.txt", header=None
)
server   = '10.175.84.61'
database = 'Solis'
username = re.search(r'username:\s*(.*)', credenciais.iloc[0, 0]).group(1)
password = re.search(r'password:\s*(.*)', credenciais.iloc[1, 0]).group(1)

connection_string = (
    'DRIVER={ODBC Driver 17 for SQL Server};'
    f'SERVER={server};DATABASE={database};UID={username};PWD={password}'
)

# Filtra apenas dados a partir desta data (deixe None para tudo)
DATA_INICIO = pd.to_datetime('2026-01-01')


def conectar():
    return pyodbc.connect(connection_string)


def ler_tabela(query: str) -> pd.DataFrame:
    with conectar() as conn:
        return pd.read_sql(query, conn)

In [10]:
print("Lendo tabela PL e cadastros ANBIMA...")
df_pl               = ler_tabela("SELECT * FROM BdTeste.CVM.CDA_PL")
df_anbima_fundo     = ler_tabela("SELECT * FROM BdTeste.Anbima.Detalhe_Fundo_Classe")
df_anbima_subclasse = ler_tabela("SELECT * FROM BdTeste.Anbima.Detalhe_Subclasse")

print("DI_Efet")

df_cdi = ler_tabela(f"SELECT * FROM GRL.Bench.DI_Efet")
df_cdi['DI_aa_ftr'] = df_cdi['DI_aa'] / 100 + 1
df_cdi['DI_ad_ftr'] = df_cdi['DI_aa_ftr']**(1/252)
df_cdi['DI_ad'] = df_cdi['DI_ad_ftr'] - 1
df_cdi['Data_Posicao'] = pd.to_datetime(df_cdi['Data_Posicao'])
df_cdi = df_cdi[['Data_Posicao', 'DI_aa', 'DI_ad', 'DI_aa_ftr', 'DI_ad_ftr']]

print("Historico Anbima")

dados_historico_anbima = ler_tabela(f"SELECT * FROM BdTeste.Anbima.Historico")
dados_historico_anbima['Data_Posicao'] = pd.to_datetime(dados_historico_anbima['Data_Posicao'])
dados_historico_anbima = dados_historico_anbima[dados_historico_anbima['Data_Posicao'] >= DATA_INICIO]
dados_historico_anbima['Codigo_Subclasse'] = np.where(dados_historico_anbima['Codigo_Classe'] == dados_historico_anbima['Codigo_Subclasse'],
                                             np.where(dados_historico_anbima['Codigo_Subclasse'].str.contains("C"),
                                             dados_historico_anbima['Codigo_Subclasse'].replace("C", "S"), 
                                             dados_historico_anbima['Codigo_Subclasse']),dados_historico_anbima['Codigo_Subclasse'])
dados_historico_anbima = dados_historico_anbima.merge(df_anbima_subclasse[['ID_CNPJ_Fundo', 'Codigo_Subclasse', 'Nome_Comercial_Subclasse']],
                                                    on=['Codigo_Subclasse', 'ID_CNPJ_Fundo'],
                                                    how='left')

df_retorno = dados_historico_anbima[['ID_CNPJ_Fundo', 'Codigo_Subclasse', 'Data_Posicao', 'PU_Cota', 'PL_Total']]



Lendo tabela PL e cadastros ANBIMA...


C:\Users\marcos.chaves\AppData\Local\Temp\ipykernel_6780\2864018712.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


DI_Efet
Historico Anbima


C:\Users\marcos.chaves\AppData\Local\Temp\ipykernel_6780\2864018712.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [ ]:
print("DI_Efet")

df_cdi = ler_tabela(f"SELECT * FROM BdTeste.GRL.Bench.DI_Efet")
df_cdi['DI_aa_ftr'] = df_cdi['DI_aa'] / 100 + 1
df_cdi['DI_ad_ftr'] = df_cdi['DI_aa_ftr']**(1/252)
df_cdi['DI_ad'] = df_cdi['DI_ad_ftr'] - 1
df_cdi = df_cdi[['Data_Posicao', 'DI_aa', 'DI_ad', 'DI_aa_ftr', 'DI_ad_ftr']]

print("Historico Anbima")

dados_historico_anbima = ler_tabela(f"SELECT * FROM BdTeste.Anbima.Historico")
dados_historico_anbima = dados_historico_anbima[dados_historico_anbima['Data_Posicao'] >= DATA_INICIO]
dados_historico_anbima['Codigo_Subclasse'] = np.where(dados_historico_anbima['Codigo_Classe'] == dados_historico_anbima['Codigo_Subclasse'],
                                             np.where(dados_historico_anbima['Codigo_Subclasse'].str.contains("C"),
                                             dados_historico_anbima['Codigo_Subclasse'].replace("C", "S")), 
                                             dados_historico_anbima['Codigo_Subclasse'])
dados_historico_anbima = dados_historico_anbima.merge(df_anbima_subclasse[['ID_CNPJ_Fundo', 'Codigo_Subclasse', 'Nome_Comercial_Subclasse']],
                                                    on=['Codigo_Subclasse', 'ID_CNPJ_Fundo'],
                                                    how='left')

df_retorno = df_retorno[['ID_CNPJ_Fundo', 'Codigo_Subclasse', 'Data_Posicao', 'PU_Cota', 'PL_Total']]

df_retorno = df_retorno.groupby(['ID_CNPJ_Fundo', 'Codigo_Subclasse']).apply(lambda x: x.sort_values('Data_Posicao'))
df_retorno['COTA_ad_ftr'] = df_retorno['PU_Cota'] / df_retorno['PU_Cota'].shift(1)
df_retorno['COTA_ad'] = df_retorno['COTA_ad_ftr'] - 1
df_retorno['COTA_aa_ftr'] = df_retorno['COTA_ad_ftr']**252
df_retorno['COTA_aa'] = df_retorno['COTA_aa_ftr'] - 1
df_retorno = df_retorno.merge(df_cdi, on='Data_Posicao', how='left')
df_retorno['Inicio'] = (df_retorno['Data_Posicao'] == df_retorno['Data_Posicao'].min()) & (df_retorno['COTA_ad_ftr'].isna())
df_retorno = df_retorno[~df_retorno['Inicio']]
df_retorno.to_excel("df_retorno.xlsx", index=False)

KeyError: 'ID_CNPJ_Fundo'

In [ ]:
df_retorno['COTA_ad_ftr'] = df_retorno['PU_Cota'] / df_retorno['PU_Cota'].shift(1)
df_retorno['COTA_ad'] = df_retorno['COTA_ad_ftr'] - 1
df_retorno['COTA_aa_ftr'] = df_retorno['COTA_ad_ftr']**252
df_retorno['COTA_aa'] = df_retorno['COTA_aa_ftr'] - 1
df_retorno = df_retorno.merge(df_cdi, on='Data_Posicao', how='left')
df_retorno['Inicio'] = (df_retorno['Data_Posicao'] == df_retorno['Data_Posicao'].min()) & (df_retorno['COTA_ad_ftr'].isna())
df_retorno = df_retorno[~df_retorno['Inicio']]